# 04. Normalization + Canonicalization Test

This notebook checks body-relative normalization and the optional analysis-space canonicalization layer in one place.

Current base normalization method:

- translation reference: frame-wise hip center
- scale reference: sequence-wise median torso length

Optional canonicalization layer:

- runs after base `norm` coordinates, not directly on raw coordinates
- preserves raw and normalized coordinates
- adds `<landmark>_canon_x/y/z` when explicitly enabled
- combines review priors such as `support_plane_alignment` and `movement_plane_alignment`
- emits a `canonicalization_report` and data-confidence summary
- remains visualization/review-only in this pass (`report_only=True`, downstream coordinate mode stays `norm`)


In [ ]:
import json

import pandas as pd
import plotly.graph_objects as go

from movement.io import load_pose_csv
from movement.config import LANDMARKS, CONNECTIONS
from movement.canonicalization import (
    CanonicalizationConfig,
    MovementPlaneAlignmentConfig,
    ProtocolHeightLateralWidthAlignmentConfig,
    apply_canonicalization,
)
from movement.floor_reference import FloorReferenceConfig
from movement.normalization import (
    normalize_pose_by_hip_torso,
    check_normalization_result,
)
from movement.visualization import (
    create_pose_animation,
    create_pose_comparison_animation,
)

from movement.pipeline import (
    ExerciseDefinitionConfig,
    NormalizationConfig,
    PipelineConfig,
    ValidationConfig,
    run_pipeline,
)


In [ ]:
csv_path = "../data/pose/sample/mediapipe_squat_synthetic.csv"

df = load_pose_csv(csv_path)

df.head()


In [ ]:
norm_df, norm_report = normalize_pose_by_hip_torso(
    df=df,
    landmarks=LANDMARKS,
)

print(json.dumps(norm_report, indent=2, ensure_ascii=False))


In [ ]:
check_report = check_normalization_result(norm_df)

print(json.dumps(check_report, indent=2, ensure_ascii=False))


In [ ]:
fig_compare = create_pose_comparison_animation(
    df=norm_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_modes=("raw", "norm"),
    names=("Raw", "Normalized"),
    title="Raw vs Normalized Pose Coordinates",
    show_text=False,
)

fig_compare.show()


## Interpretation

The raw and normalized skeletons may appear in different positions because they use different coordinate systems.

- Raw coordinates use the original pose/model coordinate space.
- Normalized coordinates use a hip-centered and torso-scaled coordinate space.

The comparison view is mainly for debugging whether both coordinate modes are available and whether the normalized skeleton behaves consistently.

Expected checks:

- `check_report["passed"]` should be `True`
- normalized hip center should be approximately zero
- median normalized torso length should be approximately 1.0


## Direct Canonicalization Test

Canonicalization starts from the normalized coordinate family and creates separate `canon` columns. In this pass the output is review-only: downstream stages continue to use `norm` coordinates.


In [ ]:
support_config = FloorReferenceConfig(
    enabled=True,
    method='support_contact_plane',
    coordinate_mode='norm',
    vertical_axis='y',
    support_landmarks=[
        'left_heel',
        'right_heel',
        'left_foot_index',
        'right_foot_index',
    ],
    diagnostic_landmarks=[
        'left_heel',
        'right_heel',
        'left_foot_index',
        'right_foot_index',
    ],
    visibility_threshold=0.7,
    max_anchor_residual_torso=0.08,
    correction_transform='rigid_rotation',
    camera_pitch_deg=0.0,
    camera_roll_deg=0.0,
    correction_strength=1.0,
    max_correction_torso=0.25,
)

movement_config = MovementPlaneAlignmentConfig(
    enabled=True,
    method='principal_motion_plane',
    fit_landmarks=[
        'left_hip',
        'left_knee',
        'left_ankle',
        'right_hip',
        'right_knee',
        'right_ankle',
    ],
    minimum_visible_landmark_ratio=0.7,
    correction_strength=0.5,
    max_rotation_deg=20.0,
    preserve_out_of_plane_residual=True,
)

protocol_height_config = ProtocolHeightLateralWidthAlignmentConfig(
    enabled=True,
    observed_height_level='H2',
    recommended_height_level='H2',
    require_height_match=True,
    correction_strength=0.3,
    max_scale_change=0.20,
    max_correction_torso=0.15,
    min_depth_offset_torso=0.05,
    visibility_threshold=0.6,
)

canonical_config = CanonicalizationConfig(
    enabled=True,
    coordinate_mode='norm',
    output_prefix='canon',
    report_only=True,
    downstream_coordinate_mode='norm',
    support_plane_alignment=support_config,
    movement_plane_alignment=movement_config,
    protocol_height_lateral_width_alignment=protocol_height_config,
)

canon_df, canon_report = apply_canonicalization(
    df=norm_df,
    landmarks=LANDMARKS,
    config=canonical_config,
)

print(json.dumps(canon_report, indent=2, ensure_ascii=False))


## Check 1: Canonical Output Columns Present


In [ ]:
assert canonical_config.report_only is True
assert canonical_config.downstream_coordinate_mode == 'norm'
assert canon_report['status'] in {'applied', 'partial'}, canon_report
assert 'support_plane_alignment' in canon_report['applied_priors']
assert 'movement_plane_alignment' in canon_report['active_priors']
assert 'protocol_height_lateral_width_alignment' in canon_report['active_priors']

for landmark in LANDMARKS:
    for axis in ['x', 'y', 'z']:
        assert f'{landmark}_canon_{axis}' in canon_df.columns
        assert f'{landmark}_norm_{axis}' in canon_df.columns
        assert f'{landmark}_{axis}' in canon_df.columns

for landmark in support_config.diagnostic_landmarks:
    assert f'{landmark}_canon_support_plane_height' in canon_df.columns

for col in [
    'canonicalization_valid',
    'canonicalization_status',
    'canonicalization_confidence',
    'canonicalization_correction_abs_frame',
]:
    assert col in canon_df.columns

print('PASS: raw/norm/canon coordinate families are present and canon is review-only')


## Check 2: Report Summary

In [ ]:
support_report = canon_report['prior_reports']['support_plane_alignment']
movement_report = canon_report['prior_reports']['movement_plane_alignment']
protocol_height_report = canon_report['prior_reports'][
    'protocol_height_lateral_width_alignment'
]
summary = pd.DataFrame([
    {
        'status': canon_report['status'],
        'data_confidence': canon_report['data_confidence']['level'],
        'report_only': canon_report['report_only'],
        'downstream_coordinate_mode': canon_report['downstream_coordinate_mode'],
        'support_status': support_report['status'],
        'movement_status': movement_report['status'],
        'movement_applied_rotation_deg': movement_report['applied_rotation_deg'],
        'movement_residual_after_p90': movement_report['out_of_plane_residual_ratio_after']['p90'],
        'protocol_height_status': protocol_height_report['status'],
        'protocol_observed_height': protocol_height_report['observed_height_level'],
        'protocol_recommended_height': protocol_height_report['recommended_height_level'],
        'protocol_anchor_height': protocol_height_report['anchor_height_level'],
        'protocol_anchor_landmarks': ', '.join(protocol_height_report['anchor_landmarks']),
        'protocol_corrected_values': protocol_height_report['num_corrected_values'],
        'protocol_far_side_report_only_values': protocol_height_report['num_far_side_report_only_values'],
        'protocol_max_scale_delta': protocol_height_report['max_scale_delta'],
        'num_anchor_points': support_report['num_anchor_points'],
        'num_anchor_frames': support_report['num_anchor_frames'],
        'max_correction_torso': canon_report['max_correction_torso'],
        'median_correction_torso': canon_report['median_correction_torso'],
        **support_report['plane_coefficients'],
    }
])
summary


## Normalized Visualization

In [ ]:
fig_norm = create_pose_animation(
    df=norm_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode='norm',
    frame_duration=100,
    height=700,
    width=950,
    show_text=False,
    title='Synthetic squat normalized coordinates',
)

fig_norm

## Canonical Visualization


In [ ]:
fig_canon = create_pose_animation(
    df=canon_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode='canon',
    frame_duration=100,
    height=700,
    width=950,
    show_text=False,
    title='Synthetic squat canonical coordinates',
)

fig_canon


## Normalized vs Canonical Comparison

This is the primary Task A gate for the existing synthetic data. Blue is the default downstream `norm` coordinate family; red is the review-only `canon` coordinate family.


In [ ]:
fig_compare = create_pose_comparison_animation(
    df=canon_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_modes=('norm', 'canon'),
    names=('Normalized', 'Canonical'),
    frame_duration=100,
    height=750,
    width=1000,
    show_text=False,
    title='Synthetic squat normalized vs canonical pose',
)

fig_compare


## Canonicalization Diagnostics

Support-plane height is a residual against the pseudo-floor prior. Movement-plane residual ratios describe how much motion remains outside the aligned review plane. These are diagnostic signals, not movement-quality deductions.


In [ ]:
fig_diag = go.Figure()

for landmark in support_config.diagnostic_landmarks:
    col = f'{landmark}_canon_support_plane_height'
    fig_diag.add_trace(
        go.Scatter(
            x=canon_df['frame'],
            y=canon_df[col],
            mode='lines',
            name=col,
        )
    )

fig_diag.add_trace(
    go.Scatter(
        x=canon_df['frame'],
        y=canon_df['canonicalization_correction_abs_frame'],
        mode='lines',
        name='canonicalization_correction_abs_frame',
        line=dict(dash='dash'),
    )
)

if 'canonicalization_lateral_width_scale_delta_frame' in canon_df.columns:
    fig_diag.add_trace(
        go.Scatter(
            x=canon_df['frame'],
            y=canon_df['canonicalization_lateral_width_scale_delta_frame'],
            mode='lines',
            name='lateral_width_scale_delta_frame',
            line=dict(dash='dot'),
        )
    )

fig_diag.update_layout(
    title='Synthetic squat canonicalization diagnostics',
    xaxis_title='Frame',
    yaxis_title='torso_length_ratio',
    height=420,
    width=950,
)

fig_diag


## Check 3: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation = ValidationConfig(enabled=True)
cfg.normalization = NormalizationConfig(enabled=True, keep_reference_columns=True)
cfg.canonicalization = canonical_config

pipe_df, pipe_report = run_pipeline(df, cfg, landmarks=LANDMARKS)

assert 'canonicalization' in pipe_report
assert pipe_report['canonicalization']['status'] in {'applied', 'partial'}
assert pipe_report['canonicalization']['report_only'] is True
assert pipe_report['canonicalization']['downstream_coordinate_mode'] == 'norm'
assert 'left_heel_canon_support_plane_height' in pipe_df.columns

print('PASS: normalization.canonicalization priors present; downstream mode remains norm')
print(json.dumps(pipe_report['canonicalization'], indent=2, ensure_ascii=False))


## Interpretation

Expected result for the clean synthetic squat: canonicalization should produce separate canon coordinate columns, provide a report with active priors, correction magnitude, residuals, and data confidence, and render a normalized-vs-canonical comparison. The normalized coordinates remain the default downstream input in this pass; canon is used only for visual review and data-confidence interpretation.
